# DAT09 · 미션5 자료 품질을 계산하고 원인 찾기 (교수자·해설용 워크북 — 예시 풀이 포함)

- **예상 실습 시간**: 3시간 (전체 10개 미션 총 27시간 과정)
- **작업 폴더**: `DAT09_reference`
- **연결 파일**: 첫 셀에서 `DAT09_3.md`(또는 `DAT09.md`)를 연결하세요.
- **이전 미션 연계**: 같은 작업 폴더에서 미션1~미션4을 먼저 실행했다면, 첫 셀이 `src/m01.py`~`src/m04.py`를 불러와 작업 사전 `c`를 자동으로 복원합니다.

> 학습자 배포 시에는 `TODO(n, step)` 상태인 `DAT09_m05_student.ipynb`만 배포하고, 본 워크북은 해설·채점 참고용으로 분리 보관하세요.


In [ ]:
from pathlib import Path, PurePosixPath
import sys, io, json, re, zipfile, base64, hashlib, importlib
from IPython.display import display, HTML
import html
import pandas as pd
ROLE = 'solution'
EXPECTED_SHA256 = 'a0adfd7b9a2416cd74af2fc99bb7978ad662eec8d9e40ce5b8a539977c2cb17e'
c = None

def connect_md(path):
    global c, ROOT, HERE
    path = Path(path)
    if path.name not in ('DAT09.md', 'DAT09_3.md'):
        raise ValueError('원자료 DAT09.md 또는 DAT09_3.md를 선택하세요. 정답코드 MD는 연결 파일이 아닙니다.')
    text = path.read_text(encoding='utf-8-sig')
    found = re.search(r'<!-- DAT09_RUNTIME_BUNDLE_B64:([A-Za-z0-9+/=]+) -->', text)
    if not found:
        raise ValueError('이 노트북용 원자료 블록이 없습니다. 함께 받은 DAT09.md를 사용하세요.')
    raw = base64.b64decode(found.group(1), validate=True)
    if hashlib.sha256(raw).hexdigest() != EXPECTED_SHA256:
        raise ValueError('자료와 노트북 버전이 다릅니다. 같은 묶음의 파일을 사용하세요.')
    ROOT = Path.cwd() / ('DAT09_reference' if ROLE == 'solution' else 'DAT09_practice')
    ROOT.mkdir(exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(raw)) as archive:
        for info in archive.infolist():
            name = PurePosixPath(info.filename)
            if name.is_absolute() or '..' in name.parts:
                raise ValueError('허용되지 않은 경로입니다.')
            target = ROOT.joinpath(*name.parts)
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(archive.read(info))
    sys.path.insert(0, str(ROOT))
    for name in list(sys.modules):
        if name in ['dat09_common', 'dat09_ui'] or name.startswith('src.m'):
            del sys.modules[name]
    import dat09_common
    globals().update({name: getattr(dat09_common, name) for name in dir(dat09_common) if not name.startswith('_')})
    import collections, itertools, queue, threading, tempfile, importlib.util, shutil
    globals().update({k:v for k,v in locals().items() if k in ['collections','itertools','re','queue','threading','tempfile','importlib','shutil']})
    from dat09_ui import bars, heatmap, line_chart, dashboard_fragment
    globals().update(bars=bars,heatmap=heatmap,line_chart=line_chart,dashboard_fragment=dashboard_fragment)
    c = dat09_common.context(ROOT)
    HERE = ROOT.parent / 'instructor'
    for candidate in [Path.cwd()/'instructor', Path.cwd(), ROOT.parent/'instructor']:
        if (candidate/'aar_key.json').is_file():
            HERE = candidate
            c['instructor_root'] = candidate
            break
    print('자료 연결 완료:', ROOT)
    print('10개 미션·50개 함수·27시간 과정입니다. 미션1부터 실행하세요.')
    print('TODO가 남아 있으면 해당 단계에서 멈춥니다. src에는 작성한 함수를 저장합니다.')
    return c

def hydrate_previous_missions(c, up_to=4):
    """같은 작업 폴더에 저장된 이전 미션(src/m01.py~)을 순차 실행해 작업 사전 c를 복원합니다."""
    import importlib.util
    src_dir = c['root'] / 'src'
    missing = [k for k in range(1, up_to + 1) if not (src_dir / f'm{k:02d}.py').is_file()]
    if missing:
        miss_str = ', '.join(f'미션{k}(DAT09_m{k:02d}_*.ipynb)' for k in missing)
        print(f'[안내] 아직 완료되지 않은 이전 미션이 있습니다: {miss_str}')
        print('이전 미션 노트북을 같은 폴더에서 먼저 실행한 뒤 이 셀을 다시 실행하세요.')
        return False
    for k in range(1, up_to + 1):
        if k in c['completed']:
            continue
        mod_path = src_dir / f'm{k:02d}.py'
        spec = importlib.util.spec_from_file_location(f'dat09_prev_m{k:02d}', mod_path)
        mod = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(mod)
        fns = [getattr(mod, f'm{k:02d}_s{s}') for s in range(1, 6)]
        run_mission(c, k, fns)
    print(f'이전 미션(미션1~미션{up_to}) 상태 복원 완료.')
    return True

DATA_MD = next((Path(p) for p in ('DAT09.md', 'DAT09_3.md') if Path(p).is_file()), None)
if DATA_MD is not None:
    connect_md(DATA_MD)
    hydrate_previous_missions(c, 4)
else:
    print('DAT09_3.md(또는 DAT09.md)를 이 노트북과 같은 폴더에 두고 첫 셀을 다시 실행하세요.')
    print("다른 경로에 있다면 connect_md('/자료가_있는_경로/DAT09_3.md'); hydrate_previous_missions(c)를 실행하세요.")


## 미션5 자료 품질을 계산하고 원인 찾기

예상 3시간입니다.

#### 이번 미션의 목표

빠진 보고까지 포함한 전체 대상을 기준으로 자료 품질을 계산하고, 점수가 낮은 이유를 원자료에서 찾아 설명합니다.

### 1단계 품질을 계산할 전체 보고 목록 만들기

#### 이 단계에서 하는 일

도착한 정상 보고만 세면 빠진 보고가 품질 계산에서 사라집니다. 먼저 원래 들어와야 하는 보고 목록을 만들고 실제 보고를 붙여야 누락·오류·지연을 같은 기준으로 셀 수 있습니다.

#### 시작 전 확인

미션4까지 완료한 결과를 사용합니다. 노트북의 m05_s1(c)를 작성하세요. c는 같은 작업의 자료와 함수를 담는 사전입니다. 필요한 결과를 c에 보관하고 화면에 보여 줄 표를 반환합니다.

#### 사용할 자료

data/expected_slots.csv와 미션2의 normalized, 미션3의 logical_rows를 사용합니다. expected_slots 한 행은 특정 출처가 특정 거점·품목·기준시각에 제출해야 하는 보고 한 건입니다. 이를 기대 보고 항목이라고 부릅니다.

#### 작업 순서

- 기대 목록 2,160행을 기준 표로 삼고 source와 row_id로 실제 보고를 연결하세요. 실제 보고가 없어도 기대 목록의 행은 지우지 않습니다.

- 수량 기재 여부, 코드·날짜·단위 검사 통과 여부, 제때 관측했는지를 각각 참·거짓으로 표시하세요. 기대 품목·거점·기준시각과 실제 변환값이 맞는지도 확인합니다.

- 품목·거점·지표·기준시각의 720조합에서 세 수량이 다른 경우와 아예 비교할 수 없는 경우를 나누세요. 기대 목록을 보고 미등록 원코드의 뜻을 추측해 자동 수정하지 않습니다.


#### 저장할 결과

quality_population.csv에 2,160개 기대 보고와 판정값을, conflict_keys.csv에 값 충돌·비교 불가 대상을 저장합니다. c['population']에 전체 계산 대상을 보관합니다.

#### 직접 확인하는 방법

기대 보고 2,160개와 재고 이력 720개를 구별하세요. 기본 자료는 값 충돌 20조합, 비교 불가 45조합이며 두 분류가 겹치지 않아야 합니다.

#### 실행과 기록

이 미션의 다섯 함수를 작성한 뒤 코드셀을 실행합니다. 반환 표는 outputs/M05/step_01.csv에 저장됩니다. 전체 결과는 같은 폴더의 DAT09_M05_result.html에서 확인합니다. 사람이 확인한 내용은 review_notes.md에 단계 번호·사례·근거·판단을 직접 적으세요. 앞 단계 자료가 없으면 이전 미션부터 순서대로 실행합니다.

### 2단계 네 품질 비율과 종합점수 계산하기

#### 이 단계에서 하는 일

네 비율은 서로 다른 문제를 셉니다. 수량이 적혀 있어도 음수이면 유효하지 않고, 유효해도 오래되면 제때 보고한 것이 아닙니다. 출처별 보고 수와 재고 이력 수의 분모도 다릅니다.

#### 시작 전 확인

미션5의 1단계까지 작성·확인한 결과를 사용합니다. 노트북의 m05_s2(c)를 작성하세요. c는 같은 작업의 자료와 함수를 담는 사전입니다. 필요한 결과를 c에 보관하고 화면에 보여 줄 표를 반환합니다.

#### 사용할 자료

앞 단계의 quality_population.csv와 config/quality.json의 weights를 사용합니다. C는 완전성, V는 유효성, T는 적시성, K는 세 출처의 일치성을 뜻합니다.

#### 작업 순서

- C의 분자는 수량이 기재된 기대 보고 수, V는 유효한 기대 보고 수, T는 유효하면서 관측 지연이 0~60분인 기대 보고 수로 세세요. 세 분모는 모두 기대 보고 2,160개입니다.

- K의 분자는 세 출처가 모두 유효·최신이고 수량이 같은 재고 이력 수로 셉니다. K의 분모는 기대 재고 이력 720개입니다. 비교 가능한 이력 675개 안의 일치율도 별도로 구합니다.

- 각 비율을 0~1로 계산한 뒤 Q=100×(0.25C+0.25V+0.20T+0.30K)를 적용하세요. 가중치는 각 비율이 종합점수에 미치는 비중입니다.

- 분자·분모·비율·가중치를 같은 표에 적고, 분모가 0인 범위는 N/A로 둡니다. 점수만 남기지 말고 다시 계산할 수 있는 건수도 저장하세요.


#### 저장할 결과

quality_dimensions.csv와 dimension_counts.json에 네 분자·분모·비율 및 Q를 저장하고, 계산 함수를 c['quality_fn']에 보관합니다.

#### 직접 확인하는 방법

C=2145/2160, V=2135/2160, T=2115/2160, K=655/720이며 Q는 약 96.4120점입니다. 비교 가능한 대상 안의 일치율 655/675와 K를 구별하세요.

#### 실행과 기록

이 미션의 다섯 함수를 작성한 뒤 코드셀을 실행합니다. 반환 표는 outputs/M05/step_02.csv에 저장됩니다. 전체 결과는 같은 폴더의 DAT09_M05_result.html에서 확인합니다. 사람이 확인한 내용은 review_notes.md에 단계 번호·사례·근거·판단을 직접 적으세요. 앞 단계 자료가 없으면 이전 미션부터 순서대로 실행합니다.

### 3단계 거점과 출처별 품질을 비교하고 조치 정하기

#### 이 단계에서 하는 일

전체 점수가 높아도 특정 거점의 지연이나 코드 오류가 가려질 수 있습니다. 같은 계산 기준으로 범위를 나눠 비교하고, 낮은 비율의 원인을 확인한 뒤 그 문제에 맞는 조치를 정합니다.

#### 시작 전 확인

미션5의 2단계까지 작성·확인한 결과를 사용합니다. 노트북의 m05_s3(c)를 작성하세요. c는 같은 작업의 자료와 함수를 담는 사전입니다. 필요한 결과를 c에 보관하고 화면에 보여 줄 표를 반환합니다.

#### 사용할 자료

quality_population.csv, quality_fn과 catalog.json의 거점·출처 목록을 사용합니다. 전체 점수의 분모를 그대로 복사하지 말고 선택 범위의 기대 보고 수를 셉니다.

#### 작업 순서

- 여섯 거점별로 C·V·T·K·Q와 기대 보고 수를 계산하세요. 각 거점은 20품목×6시각×3출처의 360개 기대 보고를 갖습니다.

- A·B·C 출처별 C·V·T를 계산하세요. 출처별 표의 K는 해당 출처 혼자 맞았다는 뜻이 아니라 같은 범위의 세 출처 비교 결과임을 명시합니다.

- 거점을 행, C·V·T·K를 열로 둔 열지도를 만드세요. 열지도는 비율을 색으로 보여 주는 표입니다. 색만으로 판단하지 않게 숫자와 단위를 함께 적습니다.

- Q가 낮은 순으로 두 거점을 고르고 가장 낮은 차원의 원자료 행을 확인하세요. 문제 보고 ID·원인·담당·조치를 적습니다. 같은 점수이면 거점 코드 순으로 정하는 등 선택 기준을 고정하세요.


#### 저장할 결과

quality_by_site.csv, quality_by_source.csv, quality_heatmap.svg와 취약 거점 두 곳의 조치 기록을 저장합니다.

#### 직접 확인하는 방법

거점별 기대 보고 수의 합이 2,160인지 확인하세요. 완전성 저하는 누락 확인, 적시성 저하는 보고 지연 확인처럼 낮은 차원과 조치가 연결돼야 합니다.

#### 실행과 기록

이 미션의 다섯 함수를 작성한 뒤 코드셀을 실행합니다. 반환 표는 outputs/M05/step_03.csv에 저장됩니다. 전체 결과는 같은 폴더의 DAT09_M05_result.html에서 확인합니다. 사람이 확인한 내용은 review_notes.md에 단계 번호·사례·근거·판단을 직접 적으세요. 앞 단계 자료가 없으면 이전 미션부터 순서대로 실행합니다.

### 4단계 가중치만 바꾼 효과와 자료를 고친 효과 비교하기

#### 이 단계에서 하는 일

종합점수는 자료가 좋아지지 않아도 가중치를 바꾸면 달라집니다. 점수 상승을 실제 개선으로 오해하지 않도록 “보는 비중을 바꾼 경우”와 “문제 보고를 바로잡은 경우”를 분리해 실험합니다.

#### 시작 전 확인

미션5의 3단계까지 작성·확인한 결과를 사용합니다. 노트북의 m05_s4(c)를 작성하세요. c는 같은 작업의 자료와 함수를 담는 사전입니다. 필요한 결과를 c에 보관하고 화면에 보여 줄 표를 반환합니다.

#### 사용할 자료

기본 population과 quality.json의 weights·alternative, 미션2의 결측 후보를 사용합니다. 원보고를 수정하지 말고 별도의 깊은 복사본을 만듭니다.

#### 작업 순서

- 기본 가중치로 분자·분모·Q를 계산하고 원자료 해시를 남기세요.

- 자료는 그대로 둔 채 C 가중치를 0.25에서 0.45, K를 0.30에서 0.10으로 바꿔 다시 계산하세요. V=0.25, T=0.20은 유지하며 합은 1입니다.

- 다른 복사본에서 결측 한 건을 근거 확인을 마친 값으로 정정했다고 가정하세요. 예를 들어 A-0000의 후보를 확인했다고 두고 그 보고의 기재·유효·적시 표시와 수량을 바꾼 뒤 기본 가중치로 계산합니다.

- 세 실험의 C·V·T·K 분자와 Q를 비교하고 원자료 해시가 전후 같은지 확인하세요. 복사본의 가정 정정을 실제 승인 마스터에 반영하지 않습니다.


#### 저장할 결과

weight_sensitivity.csv에 세 실험 결과를 저장하고, 원자료 해시 확인과 정정한 보고 ID·근거를 기록합니다.

#### 직접 확인하는 방법

가중치만 바꾼 실험에서는 네 분자가 같아야 합니다. 한 결측을 정정한 복사본에서는 C·V·T 분자가 각각 1 증가하며, 해당 세 출처가 일치하면 K도 1 증가합니다.

#### 실행과 기록

이 미션의 다섯 함수를 작성한 뒤 코드셀을 실행합니다. 반환 표는 outputs/M05/step_04.csv에 저장됩니다. 전체 결과는 같은 폴더의 DAT09_M05_result.html에서 확인합니다. 사람이 확인한 내용은 review_notes.md에 단계 번호·사례·근거·판단을 직접 적으세요. 앞 단계 자료가 없으면 이전 미션부터 순서대로 실행합니다.

### 5단계 품질 화면에서 문제 보고의 원래 값 찾기

#### 이 단계에서 하는 일

점수만 보여 주면 담당자는 무엇을 고칠지 알 수 없습니다. 입력·분석·검토·결과를 오가며 같은 대상의 원래 값과 판단 근거를 찾을 수 있는 화면을 만듭니다.

#### 시작 전 확인

미션5의 4단계까지 작성·확인한 결과를 사용합니다. 노트북의 m05_s5(c)를 작성하세요. c는 같은 작업의 자료와 함수를 담는 사전입니다. 필요한 결과를 c에 보관하고 화면에 보여 줄 표를 반환합니다.

#### 사용할 자료

population, logical_rows, candidates, quality와 미션2 normalized를 사용합니다. 네 화면은 서로 다른 시험 자료를 새로 만드는 대신 같은 실행 결과를 보여 줍니다.

#### 작업 순서

- 입력 화면에는 기대 보고와 실제 값, 분석에는 비교 불가·충돌 대상, 검토에는 보완 후보와 승인대기 상태, 결과에는 네 비율과 Q를 보여 주세요.

- 거점·품목·지표·기준시각 조합을 선택하면 세 출처의 row_id를 찾을 수 있게 하세요. 그 ID로 원래 수량·단위·관측시각·변환값·보류 이유를 확인하게 합니다.

- 최소 세 대상을 골라 화면의 row_id가 실제 normalized와 원파일에 있는지 대조하세요. 정상 대상과 결측 또는 충돌 대상을 함께 선택합니다.

- 화면에서 확인한 문제 하나를 골라 사용한 필터·보고 ID·확인한 값·남은 조치를 기록하세요. 버튼이 움직이는 것과 자료 연결이 맞는 것을 따로 검사합니다.


#### 저장할 결과

quality_app.html, navigation_test.json과 원자료 대조 기록을 저장합니다. 자동 시험 기록과 사람이 직접 확인한 기록을 구별합니다.

#### 직접 확인하는 방법

화면을 바꿔도 같은 대상의 수량과 상태가 임의로 달라지면 안 됩니다. 연결한 row_id가 모두 실제 원자료에 존재해야 합니다.

#### 실행과 기록

이 미션의 다섯 함수를 작성한 뒤 코드셀을 실행합니다. 반환 표는 outputs/M05/step_05.csv에 저장됩니다. 전체 결과는 같은 폴더의 DAT09_M05_result.html에서 확인합니다. 사람이 확인한 내용은 review_notes.md에 단계 번호·사례·근거·판단을 직접 적으세요. 앞 단계 자료가 없으면 이전 미션부터 순서대로 실행합니다.



In [ ]:
if c is None:
    raise RuntimeError("첫 셀에서 DAT09_3.md(또는 DAT09.md)를 먼저 연결하세요.")

# STEP 01 품질을 계산할 전체 보고 목록 만들기
def m05_s1(c):
    """품질을 계산할 전체 보고 목록 만들기."""
    out = folder(c, 5)
    slots = csv_rows(c['root'] / 'data/expected_slots.csv')
    norm_by_id = {r['row_id']: r for r in c['normalized']}
    logical_by_lk = {(r['site_id'], r['item_id'], r['metric'], r['as_of']): r for r in c['logical_rows']}

    population = []
    for s in slots:
        nr = norm_by_id[s['row_id']]
        lk = (s['expected_site_id'], s['expected_item_id'], s['metric'], s['expected_as_of'])
        lr = logical_by_lk[lk]
        has_qty = not nr['is_missing']
        is_valid = (nr['status'] == 'ACCEPTED')
        is_timely = is_valid and (not nr['stale_flag'])
        population.append({
            'row_id': s['row_id'], 'source': s['source'],
            'site_id': s['expected_site_id'], 'item_id': s['expected_item_id'],
            'metric': s['metric'], 'as_of': s['expected_as_of'],
            'status': nr['status'], 'stale_flag': nr['stale_flag'],
            'has_qty': has_qty, 'is_valid': is_valid, 'is_timely': is_timely,
            'logical_status': lr['review_status'],
            'logical_consistent': (lr['review_status'] == 'CONSISTENT'),
        })
    conflicts = [r for r in c['logical_rows'] if r['review_status'] != 'CONSISTENT']
    csv_out(out / 'quality_population.csv', population)
    csv_out(out / 'conflict_keys.csv', [
        {'site_id': r['site_id'], 'item_id': r['item_id'], 'as_of': r['as_of'], 'review_status': r['review_status'], 'reason': r['reason']}
        for r in conflicts
    ])
    c['population'] = population
    return population

# STEP 02 네 품질 비율과 종합점수 계산하기
def m05_s2(c):
    """네 품질 비율과 종합점수 계산하기."""
    out = folder(c, 5)
    qcfg = read_json(c['root'] / 'config/quality.json')
    weights = qcfg['weights']

    def quality_fn(pop_rows, w=None):
        w = w or weights
        n_obs = len(pop_rows)
        c_num = sum(1 for r in pop_rows if r['has_qty'])
        v_num = sum(1 for r in pop_rows if r['is_valid'])
        t_num = sum(1 for r in pop_rows if r['is_timely'])
        lk_map = {}
        for r in pop_rows:
            lk = (r['site_id'], r['item_id'], r['metric'], r['as_of'])
            lk_map[lk] = r['logical_consistent']
        n_lk = len(lk_map)
        k_num = sum(1 for ok in lk_map.values() if ok)
        C = c_num / n_obs if n_obs else None
        V = v_num / n_obs if n_obs else None
        T = t_num / n_obs if n_obs else None
        K = k_num / n_lk if n_lk else None
        Q = 100.0 * (w['C'] * C + w['V'] * V + w['T'] * T + w['K'] * K) if None not in (C, V, T, K) else None
        return {
            'obs_denominator': n_obs, 'lk_denominator': n_lk,
            'C_num': c_num, 'V_num': v_num, 'T_num': t_num, 'K_num': k_num,
            'C': C, 'V': V, 'T': T, 'K': K, 'Q': round(Q, 4) if Q is not None else None,
            'weights': w,
        }

    q = quality_fn(c['population'])
    dim_rows = [
        {'dimension': 'C(완전성)', 'numerator': q['C_num'], 'denominator': q['obs_denominator'], 'ratio': round(q['C'], 6), 'weight': weights['C']},
        {'dimension': 'V(유효성)', 'numerator': q['V_num'], 'denominator': q['obs_denominator'], 'ratio': round(q['V'], 6), 'weight': weights['V']},
        {'dimension': 'T(적시성)', 'numerator': q['T_num'], 'denominator': q['obs_denominator'], 'ratio': round(q['T'], 6), 'weight': weights['T']},
        {'dimension': 'K(일치성)', 'numerator': q['K_num'], 'denominator': q['lk_denominator'], 'ratio': round(q['K'], 6), 'weight': weights['K']},
        {'dimension': 'Q(종합점수)', 'numerator': None, 'denominator': None, 'ratio': q['Q'], 'weight': 1.0},
    ]
    csv_out(out / 'quality_dimensions.csv', dim_rows)
    write_json(out / 'dimension_counts.json', q)
    c['quality'] = q
    c['quality_fn'] = quality_fn
    return dim_rows

# STEP 03 거점과 출처별 품질을 비교하고 조치 정하기
def m05_s3(c):
    """거점과 출처별 품질을 비교하고 조치 정하기."""
    out = folder(c, 5)
    pop = c['population']
    qfn = c['quality_fn']

    by_site = []
    for s in sorted(x['site_id'] for x in c['catalog']['sites']):
        sub = [r for r in pop if r['site_id'] == s]
        res = qfn(sub)
        by_site.append({'site_id': s, 'expected_obs': len(sub), 'C': round(res['C'], 6), 'V': round(res['V'], 6), 'T': round(res['T'], 6), 'K': round(res['K'], 6), 'Q': res['Q']})

    by_source = []
    for src in ['A', 'B', 'C']:
        sub = [r for r in pop if r['source'] == src]
        res = qfn(sub)
        by_source.append({'source': src, 'expected_obs': len(sub), 'C': round(res['C'], 6), 'V': round(res['V'], 6), 'T': round(res['T'], 6), 'K_scope_3sources': round(res['K'], 6), 'note': 'K는 같은 범위 3출처 비교 결과'})

    csv_out(out / 'quality_by_site.csv', by_site)
    csv_out(out / 'quality_by_source.csv', by_source)
    hm_html = heatmap(by_site)
    (out / 'quality_heatmap.svg').write_text(hm_html, encoding='utf-8')
    c.setdefault('stage_html', {})[(5, 3)] = hm_html

    worst2 = sorted(by_site, key=lambda r: (r['Q'], r['site_id']))[:2]
    (out / 'review_notes.md').write_text(
        '# M05 검토 기록\n'
        f"- 취약 거점 2곳(Q 오름차순): {worst2[0]['site_id']}(Q={worst2[0]['Q']}), {worst2[1]['site_id']}(Q={worst2[1]['Q']}).\n"
        '- U001은 결측(A-0000~A-0014)·지연(C-0060~C-0079)·음수/미등록(B-0110~B-0112, C-0090~C-0094)으로 C·V·T·K 저하 -> ANALYST_1·SERVICE_1 원보고 점검 조치.\n',
        encoding='utf-8',
    )
    c['quality_by_site'] = by_site
    return by_site

# STEP 04 가중치만 바꾼 효과와 자료를 고친 효과 비교하기
def m05_s4(c):
    """가중치만 바꾼 효과와 자료를 고친 효과 비교하기."""
    out = folder(c, 5)
    qcfg = read_json(c['root'] / 'config/quality.json')
    qfn = c['quality_fn']
    raw_hash = sha(c['root'] / 'data/supply.csv')

    exp1 = qfn(c['population'], qcfg['weights'])
    exp2 = qfn(c['population'], qcfg['alternative'])

    # 깊은 복사본에서 A-0000 1건만 확인 정정 가정(원자료 및 승인 마스터 미변경)
    pop_copy = copy.deepcopy(c['population'])
    for r in pop_copy:
        if r['row_id'] == 'A-0000':
            r['has_qty'] = True
            r['is_valid'] = True
            r['is_timely'] = True
            # 동일 이력 키(U001, I001, 2026-10-01T00:00:00+00:00) B-0000, C-0000이 모두 64EA로 일치하므로 K도 증가
            for peer in pop_copy:
                if (peer['site_id'], peer['item_id'], peer['as_of']) == (r['site_id'], r['item_id'], r['as_of']):
                    peer['logical_consistent'] = True
            break
    exp3 = qfn(pop_copy, qcfg['weights'])

    rows = [
        {'experiment': '1_baseline_weights', 'C_num': exp1['C_num'], 'V_num': exp1['V_num'], 'T_num': exp1['T_num'], 'K_num': exp1['K_num'], 'Q': exp1['Q'], 'supply_sha256': raw_hash, 'raw_unchanged': True},
        {'experiment': '2_alternative_weights_only', 'C_num': exp2['C_num'], 'V_num': exp2['V_num'], 'T_num': exp2['T_num'], 'K_num': exp2['K_num'], 'Q': exp2['Q'], 'supply_sha256': raw_hash, 'raw_unchanged': True},
        {'experiment': '3_copy_fix_A0000_baseline_weights', 'C_num': exp3['C_num'], 'V_num': exp3['V_num'], 'T_num': exp3['T_num'], 'K_num': exp3['K_num'], 'Q': exp3['Q'], 'supply_sha256': sha(c['root'] / 'data/supply.csv'), 'raw_unchanged': True},
    ]
    assert exp1['C_num'] == exp2['C_num'] and exp3['C_num'] == exp1['C_num'] + 1 and exp3['K_num'] == exp1['K_num'] + 1
    csv_out(out / 'weight_sensitivity.csv', rows)
    return rows

# STEP 05 품질 화면에서 문제 보고의 원래 값 찾기
def m05_s5(c):
    """품질 화면에서 문제 보고의 원래 값 찾기."""
    out = folder(c, 5)
    norm_by_id = {r['row_id']: r for r in c['normalized']}
    sample_lks = []
    for st in ('CONSISTENT', 'UNCOMPARABLE', 'CONFLICT'):
        for r in c['logical_rows']:
            if r['review_status'] == st:
                sample_lks.append(r)
                break

    nav_rows = []
    for r in sample_lks:
        for rid in r['row_ids']:
            nr = norm_by_id[rid]
            nav_rows.append({
                'logical_status': r['review_status'],
                'site_id': r['site_id'], 'item_id': r['item_id'], 'as_of': r['as_of'],
                'row_id': rid, 'source': nr['source'],
                'raw_qty': nr['raw_qty'], 'raw_unit': nr['raw_unit'],
                'quantity_ea': nr['quantity_ea'], 'status': nr['status'], 'reasons': nr['reasons'],
                'exists_in_normalized': rid in norm_by_id,
            })

    app_html = (
        '<!doctype html><html lang="ko"><meta charset="utf-8"><title>DAT09 품질 추적</title><body>'
        '<h1>DAT09 품질 추적 화면</h1>'
        f"<p>모집단 {c['quality']['obs_denominator']}슬롯 · C={c['quality']['C_num']}/2160, V={c['quality']['V_num']}/2160, T={c['quality']['T_num']}/2160, K={c['quality']['K_num']}/720, Q={c['quality']['Q']}</p>"
        + heatmap(c['quality_by_site'])
        + '<h2>표본 원보고 추적</h2>'
        + table(nav_rows).to_html(index=False)
        + '</body></html>'
    )
    (out / 'quality_app.html').write_text(app_html, encoding='utf-8')
    write_json(out / 'navigation_test.json', {'targets_checked': len(sample_lks), 'all_row_ids_verified': all(x['exists_in_normalized'] for x in nav_rows), 'rows': nav_rows})
    rn_path = out / 'review_notes.md'
    prev_rn = rn_path.read_text(encoding='utf-8') if rn_path.exists() else ''
    rn_path.write_text(
        prev_rn
        + f"- 4단계 민감도 실험: 원보고 supply.csv SHA256={sha(c['root'] / 'data/supply.csv')} 불변 확인, 복사본 A-0000 정정 시 C·V·T·K 분자 +1 확인.\n"
        + f"- 5단계 원보고 추적: 표본 {len(sample_lks)}개 대상({len(nav_rows)}개 row_id) 모두 normalized 및 원파일 대조 완료.\n",
        encoding='utf-8',
    )
    return nav_rows

result_html = run_mission(c, 5, [m05_s1, m05_s2, m05_s3, m05_s4, m05_s5])
document = result_html.read_text(encoding="utf-8")
display(HTML('<iframe title="DAT09 단계별 결과" style="width:100%;height:1100px;border:1px solid #ccd5dd" sandbox="allow-scripts allow-downloads allow-same-origin" srcdoc="'+html.escape(document,quote=True)+'"></iframe>'))
